In [1]:
import jax
import jax.numpy as jnp
from jax import random, pmap, jit
from functools import partial
import numpy as np
import time
import math
import os

# Verificación de entorno Multi-GPU
num_gpus = jax.device_count()
assert num_gpus >= 2, f"Se requieren al menos 2 GPUs, detectadas: {num_gpus}"
jax.config.update("jax_enable_x64", False) # Mantener FP32 para velocidad

# ==========================================
# PARÁMETROS FIJOS (GLOBALES)
# ==========================================
NUM_ESTADOS = 4
LOTE_ID = 5        # Cambia esto en cada corrida (0, 1, 2, 3...)
TOTAL_EXPERIMENTO = 100 # El total de estados que planeas correr en todo tu proyecto

REPLICAS_POR_ESTADO = 100
MCS_TERM = 10000
MCS_MEAS = 2000
MCS_MAX = MCS_TERM + MCS_MEAS

Rd = 18.3 
L = 5
So = 1.0
Ti = 20.0
Tf = 0.0
dT = -0.1
kB = 0.086173404
L2 = int(Rd) + 1
Ntot_grid = (2*L2 + 1) * (2*L2 + 1) * (L + 2) 

from scipy.stats import qmc # Asegúrate de importar esto arriba

# ==========================================
# GENERACIÓN DE PARÁMETROS (QMC Halton)
# ==========================================
# 1. Generar la secuencia COMPLETA para asegurar la cobertura perfecta
sampler = qmc.Halton(d=3, scramble=True, seed=42)
muestra_completa = sampler.random(n=TOTAL_EXPERIMENTO)
# 2. Extraer el bloque específico para esta corrida
inicio_idx = LOTE_ID * NUM_ESTADOS
fin_idx = inicio_idx + NUM_ESTADOS
# Validación de seguridad
assert fin_idx <= TOTAL_EXPERIMENTO, f"No hay suficientes estados en el diseño. Intentas acceder hasta el índice {fin_idx}, pero el total es {TOTAL_EXPERIMENTO}."
muestra_lote = muestra_completa[inicio_idx:fin_idx]
def mapear_hex(u):
    if u < 0.033: return 0.0 + (u / 0.033) * 0.07
    elif u < 0.883: return 0.07 + ((u - 0.033) / 0.85) * (1.0 - 0.07)
    else: return 1.0 + ((u - 0.883) / 0.117) * 0.25
def mapear_kdm(u):
    if u < 0.85: return 0.45 + (u / 0.85) * (1.4 - 0.45)
    else: return 1.4 + ((u - 0.85) / 0.15) * (1.6 - 1.4)
def mapear_kan(u):
    return u * 0.6 

# 4. Asignar los valores a las listas base
Hex_base  = [round(mapear_hex(u[0]), 2) for u in muestra_lote]
kDM_base  = [round(mapear_kdm(u[1]), 2) for u in muestra_lote]
Kan1_base = [round(mapear_kan(u[2]), 2) for u in muestra_lote]
Jex_base  = [1.0 for _ in range(NUM_ESTADOS)]
Kan1_base = [round(np.random.uniform(0.0, 0.6), 2) for _ in range(NUM_ESTADOS)]
gamma_base= [90.0 for _ in range(NUM_ESTADOS)] 
gamma_rad_base = [math.radians(g) for g in gamma_base]
print(f"Ejecutando LOTE {LOTE_ID}: Estados del {inicio_idx} al {fin_idx - 1}")

# Preparación de tensores divididos para jax.pmap (Num_GPUs, Replicas_por_GPU, ...)
estados_por_gpu = NUM_ESTADOS // num_gpus
batch_por_gpu = estados_por_gpu * REPLICAS_POR_ESTADO

def format_param(base_list):
    arr = np.array(base_list).repeat(REPLICAS_POR_ESTADO)
    return arr.reshape(num_gpus, batch_por_gpu, 1, 1, 1)

Jex_pmap = format_param(Jex_base)
kDM_pmap = format_param(kDM_base)
Kan1_pmap = format_param(Kan1_base)
Hex_pmap = format_param(Hex_base)
gamma_pmap = format_param(gamma_rad_base)
Eo_pmap = (Jex_pmap * Ntot_grid * 6 * So**2 + kDM_pmap * Ntot_grid * 6 * So**2 + Hex_pmap * Ntot_grid * So).reshape(num_gpus, batch_por_gpu)

# ==========================================
# PRECALCULO DE MÁSCARAS (CPU -> JAX)
# ==========================================
x_coords = np.arange(-L2, L2 + 1).reshape(1, 1, 1, -1)
y_coords = np.arange(-L2, L2 + 1).reshape(1, 1, -1, 1)
z_coords = np.arange(1, L + 1).reshape(1, -1, 1, 1)

radii_sq = x_coords**2 + y_coords**2
disco_mask_2d = (radii_sq <= Rd**2)

# Equivalente exacto de .expand(1, L, 2*L2 + 1, 2*L2 + 1)
disco_mask_3d = np.broadcast_to(disco_mask_2d, (1, L, 2*L2 + 1, 2*L2 + 1))

checkerboard = (x_coords + y_coords + z_coords) % 2

mask_even_np = ((checkerboard == 0) & disco_mask_3d)[..., np.newaxis]
mask_odd_np  = ((checkerboard == 1) & disco_mask_3d)[..., np.newaxis]
disco_mask_np = disco_mask_3d[..., np.newaxis]

N_spins = int(disco_mask_3d.sum()) 

# Transferir a dispositivos JAX
mask_even_jnp = jnp.array(mask_even_np)
mask_odd_jnp = jnp.array(mask_odd_np)
disco_mask_jnp = jnp.array(disco_mask_np)
# ==========================================
# FUNCIONES JAX COMPILADAS
# ==========================================
def compute_energy_components(spins, mask_vec, kDM, Jex, Kan1, Hex_mag, gamma_rad, So):
    S_xp = jnp.roll(spins, -1, axis=3); S_xm = jnp.roll(spins, 1, axis=3)
    S_yp = jnp.roll(spins, -1, axis=2); S_ym = jnp.roll(spins, 1, axis=2)
    S_zp = jnp.roll(spins, -1, axis=1); S_zm = jnp.roll(spins, 1, axis=1)
    S_zp = S_zp.at[:, -1, :, :, :].set(0.0)
    S_zm = S_zm.at[:, 0, :, :, :].set(0.0)
    sum_vecinos = S_xp + S_xm + S_yp + S_ym + S_zp + S_zm
    
    E_exc = -Jex * jnp.sum(spins * sum_vecinos, axis=-1) * 0.5
    E_an1 = (Kan1 / So**4) * ((spins[...,0]*spins[...,1])**2 + (spins[...,0]*spins[...,2])**2 + (spins[...,1]*spins[...,2])**2)
    dm_x = spins[...,1]*(S_xp[...,2] - S_xm[...,2]) - spins[...,2]*(S_xp[...,1] - S_xm[...,1])
    dm_y = spins[...,2]*(S_yp[...,0] - S_ym[...,0]) - spins[...,0]*(S_yp[...,2] - S_ym[...,2])
    E_dm = -kDM * (dm_x + dm_y) * 0.5
    E_hex = -Hex_mag * (spins[...,0]*jnp.cos(gamma_rad) + spins[...,2]*jnp.sin(gamma_rad))
    
    mask = jnp.squeeze(mask_vec, axis=-1)
    return (jnp.sum(E_exc * mask, axis=(1,2,3)),
            jnp.sum(E_an1 * mask, axis=(1,2,3)),
            jnp.sum(E_dm * mask, axis=(1,2,3)),
            jnp.sum(E_hex * mask, axis=(1,2,3)))

def metropolis_substep(spins, key, beta, mask_vec, kDM, Jex, Kan1, Hex_mag, gamma_rad):
    S_xp = jnp.roll(spins, -1, axis=3); S_xm = jnp.roll(spins, 1, axis=3)
    S_yp = jnp.roll(spins, -1, axis=2); S_ym = jnp.roll(spins, 1, axis=2)
    S_zp = jnp.roll(spins, -1, axis=1); S_zm = jnp.roll(spins, 1, axis=1)
    S_zp = S_zp.at[:, -1, :, :, :].set(0.0)
    S_zm = S_zm.at[:, 0, :, :, :].set(0.0)
    
    key_t, key_a = random.split(key)
    spins_trial = random.normal(key_t, spins.shape)
    spins_trial = spins_trial / jnp.linalg.norm(spins_trial, axis=-1, keepdims=True) * So
    spins_trial = jnp.where(mask_vec, spins_trial, spins)
    dS = spins_trial - spins
    
    sum_vecinos = S_xp + S_xm + S_yp + S_ym + S_zp + S_zm
    dE_exc = -Jex * jnp.sum(dS * sum_vecinos, axis=-1)
    E_an1_t = (Kan1 / So**4) * ((spins_trial[...,0]*spins_trial[...,1])**2 + (spins_trial[...,0]*spins_trial[...,2])**2 + (spins_trial[...,1]*spins_trial[...,2])**2)
    E_an1_s = (Kan1 / So**4) * ((spins[...,0]*spins[...,1])**2 + (spins[...,0]*spins[...,2])**2 + (spins[...,1]*spins[...,2])**2)
    dE_an1 = E_an1_t - E_an1_s
    d_dm_x = dS[...,1]*(S_xp[...,2] - S_xm[...,2]) - dS[...,2]*(S_xp[...,1] - S_xm[...,1])
    d_dm_y = dS[...,2]*(S_yp[...,0] - S_ym[...,0]) - dS[...,0]*(S_yp[...,2] - S_ym[...,2])
    dE_dm = -kDM * (d_dm_x + d_dm_y)
    dE_hex = -Hex_mag * (dS[...,0]*jnp.cos(gamma_rad) + dS[...,2]*jnp.sin(gamma_rad))
    delta_E = dE_exc + dE_an1 + dE_dm + dE_hex
    
    rand_tensor = random.uniform(key_a, delta_E.shape)
    accept = (delta_E < 0) | (rand_tensor < jnp.exp(-delta_E * beta))
    accept_mask = jnp.expand_dims(accept, -1) & mask_vec
    
    return jnp.where(accept_mask, spins_trial, spins)

# Bucle compilado en GPU de termalización pura
def term_loop(spins, key, beta, mask_even, mask_odd, kDM, Jex, Kan1, Hex_mag, gamma_rad):
    def body_fn(i, val):
        s, k = val
        k, k1, k2 = random.split(k, 3)
        s = metropolis_substep(s, k1, beta, mask_even, kDM, Jex, Kan1, Hex_mag, gamma_rad)
        s = metropolis_substep(s, k2, beta, mask_odd, kDM, Jex, Kan1, Hex_mag, gamma_rad)
        return s, k
    return jax.lax.fori_loop(0, MCS_TERM, body_fn, (spins, key))

# Bucle compilado en GPU que acumula mediciones paso a paso sin escribir en disco
def meas_loop(spins, key, beta, mask_even, mask_odd, mask_disco, kDM, Jex, Kan1, Hex_mag, gamma_rad, Eo_batch):
    def step_fn(val, _):
        s, k = val
        k, k1, k2 = random.split(k, 3)
        s = metropolis_substep(s, k1, beta, mask_even, kDM, Jex, Kan1, Hex_mag, gamma_rad)
        s = metropolis_substep(s, k2, beta, mask_odd, kDM, Jex, Kan1, Hex_mag, gamma_rad)
        
        e_jex, e_kan, e_kdm, e_hex = compute_energy_components(s, mask_disco, kDM, Jex, Kan1, Hex_mag, gamma_rad, So)
        Ene_batch = e_jex + e_kan + e_kdm + e_hex 
        M_vec = jnp.sum(s * mask_disco, axis=(1,2,3))
        M_norm = jnp.linalg.norm(M_vec, axis=-1) / N_spins
        
        # Guardar en el historial (gestionado internamente por lax.scan)
        metrics = (Ene_batch / Eo_batch, M_norm, e_jex / Eo_batch, e_kan / Eo_batch, e_kdm / Eo_batch, e_hex / Eo_batch)
        return (s, k), metrics
    return jax.lax.scan(step_fn, (spins, key), None, length=MCS_MEAS)

# ==========================================
# PARALELISMO NATIVO MULTI-GPU (PMAP)
# ==========================================
# Se mapean las variables con prefijo pmap a lo largo del eje 0 (num_gpus).
@partial(pmap, in_axes=(0, 0, None, None, None, None, 0, 0, 0, 0, 0, 0))
def step_temperature_multi_gpu(spins, key, beta, mask_even, mask_odd, mask_disco, kDM, Jex, Kan1, Hex_mag, gamma_rad, Eo_batch):
    spins, key = term_loop(spins, key, beta, mask_even, mask_odd, kDM, Jex, Kan1, Hex_mag, gamma_rad)
    (spins, key), metrics = meas_loop(spins, key, beta, mask_even, mask_odd, mask_disco, kDM, Jex, Kan1, Hex_mag, gamma_rad, Eo_batch)
    return spins, key, metrics

# ==========================================
# BUCLE PRINCIPAL (PYTHON / CPU)
# ==========================================
if __name__ == '__main__':
    print(f"Total Estados: {NUM_ESTADOS} | Átomos: {N_spins} | Replicas por estado: {REPLICAS_POR_ESTADO} | GPUs detectadas: {num_gpus}")
    start_time = time.time()
    
    # Inicialización de Espines y Claves Aleatorias para cada GPU
    key = random.PRNGKey(42)
    keys_pmap = random.split(key, num_gpus)
    
    spins_pmap = random.normal(key, (num_gpus, batch_por_gpu, L, 2*L2 + 1, 2*L2 + 1, 3))
    spins_pmap = spins_pmap / jnp.linalg.norm(spins_pmap, axis=-1, keepdims=True) * So
    spins_pmap = spins_pmap * disco_mask_jnp # Enmascarar base
    

    # Almacenamiento RAM (Diccionarios listos para NumPy)
    datos_globales = {i: {"resultados": [], "energias": [], "evolucion": []} for i in range(NUM_ESTADOS)}
    Cv_prev = np.zeros(NUM_ESTADOS)
    Chi_prev = np.zeros(NUM_ESTADOS)

    T_current = Ti
    while T_current >= Tf:
        beta = 1.0 / (kB * T_current) if T_current > 0 else 1e9
        
        # LANZAMIENTO A AMBAS GPUS SIMULTANEAMENTE
        spins_pmap, keys_pmap, metrics = step_temperature_multi_gpu(
            spins_pmap, keys_pmap, beta, mask_even_jnp, mask_odd_jnp, disco_mask_jnp, 
            kDM_pmap, Jex_pmap, Kan1_pmap, Hex_pmap, gamma_pmap, Eo_pmap
        )
        
        # Recuperar datos a CPU: la tupla `metrics` contiene arrays de forma (num_gpus, MCS_MEAS, batch_por_gpu)
        E_hist, M_hist, ej_hist, eka_hist, ekd_hist, eh_hist = [np.array(m) for m in metrics]
        
        # Procesamiento Estadístico en CPU
        E_mean = np.mean(E_hist, axis=1).reshape(NUM_ESTADOS, REPLICAS_POR_ESTADO)
        M_mean = np.mean(M_hist, axis=1).reshape(NUM_ESTADOS, REPLICAS_POR_ESTADO)
        E_sq_mean = np.mean(E_hist**2, axis=1).reshape(NUM_ESTADOS, REPLICAS_POR_ESTADO)
        M_sq_mean = np.mean(M_hist**2, axis=1).reshape(NUM_ESTADOS, REPLICAS_POR_ESTADO)
        
        E_j_mean = np.mean(ej_hist, axis=1).reshape(NUM_ESTADOS, REPLICAS_POR_ESTADO).mean(axis=1)
        E_ka_mean = np.mean(eka_hist, axis=1).reshape(NUM_ESTADOS, REPLICAS_POR_ESTADO).mean(axis=1)
        E_kd_mean = np.mean(ekd_hist, axis=1).reshape(NUM_ESTADOS, REPLICAS_POR_ESTADO).mean(axis=1)
        E_h_mean = np.mean(eh_hist, axis=1).reshape(NUM_ESTADOS, REPLICAS_POR_ESTADO).mean(axis=1)

        E_promedio = E_mean.mean(axis=1)
        M_promedio = M_mean.mean(axis=1)
        
        if T_current > 0.1:
            Cv = np.mean((E_sq_mean - E_mean**2) / (kB * T_current**2), axis=1)
            Chi = np.mean((M_sq_mean - M_mean**2) / (kB * T_current), axis=1)
            Cv_prev, Chi_prev = Cv, Chi
        else:
            Cv, Chi = Cv_prev, Chi_prev
            
        print(f"Completado T: {T_current:.2f} K")
        
        # Guardar resultados e imágenes reducidas de espines en RAM
        spins_cpu = np.array(spins_pmap).reshape(NUM_ESTADOS, REPLICAS_POR_ESTADO, L, 2*L2+1, 2*L2+1, 3)
        
        for i_estado in range(NUM_ESTADOS):
            datos_globales[i_estado]["resultados"].append([T_current, E_promedio[i_estado], Cv[i_estado], M_promedio[i_estado], Chi[i_estado]])
            datos_globales[i_estado]["energias"].append([T_current, E_ka_mean[i_estado], E_h_mean[i_estado], E_j_mean[i_estado], E_kd_mean[i_estado]])
            
            # Elegir 5 índices aleatorios distintos para evitar sesgo espacial o de estado
            indices_aleatorios = np.random.choice(REPLICAS_POR_ESTADO, size=5, replace=False)
            
            for idx_rep in indices_aleatorios:
                datos_globales[i_estado]["evolucion"].append({
                    'T': T_current, 
                    'estado': int(idx_rep + 1), # Casteo a int nativo para evitar problemas de serialización
                    'spins': spins_cpu[i_estado, idx_rep][disco_mask_np[0, ..., 0]]
                })
        
        T_current = round(T_current + dT, 2)

    # ==========================================
    # ESCRITURA EN DISCO AL FINALIZAR
    # ==========================================
    print("Simulación terminada. Volcando a disco (.npz)...")
    for i_estado in range(NUM_ESTADOS):
        nombre = f"est{i_estado}_jex={Jex_base[i_estado]:.2f}_kdm={kDM_base[i_estado]:.2f}_kan={Kan1_base[i_estado]:.2f}_hex={Hex_base[i_estado]:.2f}_gamma={gamma_base[i_estado]:.0f}"
        
        np.savez_compressed(
            f"Datos_{nombre}.npz",
            resultadosVsT=np.array(datos_globales[i_estado]["resultados"]),
            energiasVsT=np.array(datos_globales[i_estado]["energias"]),
            evolucion=np.array(datos_globales[i_estado]["evolucion"], dtype=object)
        )
        
    print(f"Pipeline multi-GPU completado. Tiempo total: {(time.time() - start_time)/3600:.2f} hrs.")

Ejecutando LOTE 5: Estados del 20 al 23
Total Estados: 4 | Átomos: 5245 | Replicas por estado: 100 | GPUs detectadas: 2
Completado T: 20.00 K
Completado T: 19.90 K
Completado T: 19.80 K
Completado T: 19.70 K
Completado T: 19.60 K
Completado T: 19.50 K
Completado T: 19.40 K
Completado T: 19.30 K
Completado T: 19.20 K
Completado T: 19.10 K
Completado T: 19.00 K
Completado T: 18.90 K
Completado T: 18.80 K
Completado T: 18.70 K
Completado T: 18.60 K
Completado T: 18.50 K
Completado T: 18.40 K
Completado T: 18.30 K
Completado T: 18.20 K
Completado T: 18.10 K
Completado T: 18.00 K
Completado T: 17.90 K
Completado T: 17.80 K
Completado T: 17.70 K
Completado T: 17.60 K
Completado T: 17.50 K
Completado T: 17.40 K
Completado T: 17.30 K
Completado T: 17.20 K
Completado T: 17.10 K
Completado T: 17.00 K
Completado T: 16.90 K
Completado T: 16.80 K
Completado T: 16.70 K
Completado T: 16.60 K
Completado T: 16.50 K
Completado T: 16.40 K
Completado T: 16.30 K
Completado T: 16.20 K
Completado T: 16.10 K
